## Address overflow

When we were developing cross entropy kernel, we were able to pass 95% of the test but 5% of them failed. What can be the reason?

**Hint: in those 5%, the input size is larger.**

### Ensure you are using GPU by running `nvidia-smi`

In [ ]:
!nvidia-smi

Wed Aug 21 18:13:17 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   51C    P0              27W /  70W |   8121MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

### Original implementation

In [ ]:
import triton
import torch
import triton.language as tl

@triton.jit
def liger_cross_entropy_kernel(
    X_ptr,
    X_stride,
    Y_ptr,
    Y_stride,
    loss_ptr,
    loss_stride,
    n_cols,
    n_non_ignore,
    ignore_index,
    BLOCK_SIZE: tl.constexpr,
):
    program_id = tl.program_id(0)

    Y_ptr += program_id * Y_stride
    y = tl.load(Y_ptr)

    # skip the remaining part


In [ ]:

MAX_FUSED_SIZE = 65536

class LigerCrossEntropyFunction(torch.autograd.Function):

    @staticmethod
    def forward(ctx, _input, target, ignore_index):
        BT, V = _input.shape
        n_rows = BT

        BLOCK_SIZE = min(MAX_FUSED_SIZE, triton.next_power_of_2(V))

        # unreduced loss
        loss_1d = torch.zeros(n_rows, dtype=_input.dtype, device=_input.device)

        n_non_ignore = (target != ignore_index).sum().item()

        # ensure _input and target are contiguous in the last dimension
        if _input.stride(-1) != 1:
            _input = _input.contiguous()
        if target.stride(-1) != 1:
            target = target.contiguous()

        # Here we use a trick to store X_ptr gradient in X_ptr so we can save memory
        liger_cross_entropy_kernel[(n_rows,)](
            X_ptr=_input,
            X_stride=_input.stride(-2),
            Y_ptr=target,
            Y_stride=target.stride(-1),  # always 1
            loss_ptr=loss_1d,
            loss_stride=loss_1d.stride(-1),  # always 1
            n_cols=V,
            n_non_ignore=n_non_ignore,
            ignore_index=ignore_index,
            BLOCK_SIZE=BLOCK_SIZE,
            # TODO: 32 seems to give the best performance
            # Performance is quite sentitive to num_warps
            num_warps=32,
        )

        loss = torch.sum(loss_1d) / n_non_ignore

        return loss

## Failing with illegal memory address

However, on Google Collab, I cannot reproduce the errors. Let's pretend it does trigger the error.

In [ ]:
B, T, V = 4, 8192, 128000

_input = torch.zeros(B * T, V, device="cuda", dtype=torch.bool) # the dtype should be 16 or 32, but the capacity is not enough on collab
target = torch.zeros(B * T, device="cuda", dtype=torch.bool)
ignore_index = -100

loss = LigerCrossEntropyFunction.apply(_input, target, ignore_index)

## Why illegal memory address?

`program_id` by default is stored in `int32`, look at this line

```python
Y_ptr += program_id * Y_stride
```

We time `program_id` by `Y_stride`. If B = 4, T = 8192, V = 128000, `program_id * Y_stride` can be 4_194_304_000, but max int32 is only 2_147_483_647!! Therefore, the value will become negative and step on illegal address!

### The fix is to add a explicit conversion to int64

In [ ]:
import triton
import torch
import triton.language as tl

@triton.jit
def liger_cross_entropy_kernel(
    X_ptr,
    X_stride,
    Y_ptr,
    Y_stride,
    loss_ptr,
    loss_stride,
    n_cols,
    n_non_ignore,
    ignore_index,
    BLOCK_SIZE: tl.constexpr,
):
    # https://github.com/triton-lang/triton/issues/1058
    # If B*T*V is too large, program_id * stride will overflow out of int32, so we convert to int64
    program_id = tl.program_id(0).to(tl.int64)

    # 1. Load Y_ptr first because if the target is ignore_index, we can return right away
    Y_ptr += program_id * Y_stride
    y = tl.load(Y_ptr)

B, T, V = 4, 8192, 128000

_input = torch.zeros(B * T, V, device="cuda", dtype=torch.bool)
target = torch.zeros(B * T, device="cuda", dtype=torch.bool)
ignore_index = -100

loss = LigerCrossEntropyFunction.apply(_input, target, ignore_index)

OutOfMemoryError: CUDA out of memory. Tried to allocate 3.91 GiB. GPU 